# Explore analytics HTTP API

Call **analytics** (portfolio universe + analyze + report workflows).

**Prerequisites**
- Run analytics with uvicorn on a port you choose; this notebook defaults to **`http://localhost:8010`** (Compose may not publish a port yet).
- Set `ANALYTICS_BASE_URL` if you use another host/port.
- Venv: `make venv-service SERVICE=analytics`; `pip install jupyterlab ipykernel httpx` if needed.
- Downstream: datalake Postgres (or parquet mode) must match server config.

**Endpoints:** `GET /health`, `POST /portfolio/universe` (minimal payload below).

Add more `.ipynb` files in this directory for ad-hoc experiments; keep `explore.ipynb` as the shared template.

In [ ]:
import json
import os

import httpx

BASE_URL = os.environ.get("ANALYTICS_BASE_URL", "http://localhost:8010").rstrip("/")
print("BASE_URL =", BASE_URL)

In [ ]:
with httpx.Client(timeout=60.0) as client:
    r = client.get(f"{BASE_URL}/health")
    r.raise_for_status()
    print(json.dumps(r.json(), indent=2))

In [ ]:
payload = {
    "symbols": ["AAPL", "MSFT"],
    "source": {"mode": "canonical_tables"},
}

with httpx.Client(timeout=60.0) as client:
    r = client.post(f"{BASE_URL}/portfolio/universe", json=payload)
    if r.status_code >= 400:
        print(r.status_code, r.text)
    else:
        r.raise_for_status()
        print(json.dumps(r.json(), indent=2))

## Next steps

- Chain `POST /portfolio/analyze` with the returned `job_id`, then `GET /portfolio/report/{job_id}`.
- See service README and `app/models.py` for full request shapes.